In [0]:
from delta import configure_spark_with_delta_pip, DeltaTable
from pyspark.sql import SparkSession

In [0]:
builder = (SparkSession.builder
           .appName("optimize-delta-table")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [0]:
%load_ext sparksql_magic
%config SparkSql.limit=20

In [0]:
# For PySpark:
df = spark.read.format("delta").load("/opt/workspace/data/delta_lake/netflix_titles")

In [0]:
df.show(5)

In [0]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/netflix_titles"

### Optimize performance with file management - Compaction (bin-packing)

In [0]:
deltaTable = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/netflix_titles")  
# For Hive metastore-based tables: deltaTable = DeltaTable.forName(spark, tableName)
deltaTable.optimize().executeCompaction()

In [0]:
%%sparksql
-- Optimizes the path-based Delta Lake table
OPTIMIZE "/opt/workspace/data/delta_lake/netflix_titles" 

### Data skipping - Z-Ordering (multi-dimensional clustering)

In [0]:
deltaTable = DeltaTable.forPath(spark, "/opt/workspace/data/delta_lake/netflix_titles")  # path-based table
# For Hive metastore-based tables: deltaTable = DeltaTable.forName(spark, tableName)
deltaTable.optimize().executeZOrderBy("country")

In [0]:
%%sparksql
OPTIMIZE "/opt/workspace/data/delta_lake/netflix_titles" ZORDER BY (country)

In [0]:
%%sparksql
DESCRIBE HISTORY "/opt/workspace/data/delta_lake/netflix_titles"

### Partitioning

In [0]:
# Read the data as a DataFrame
df = (spark.read.format("json")
      .option("multiLine", "true")
      .load("../data/nobel_prizes.json"))

In [0]:
df.printSchema()

In [0]:
# Write the data to a Delta Lake table with partitioning
(df.write.format("delta")
 .mode("overwrite")
 .partitionBy("year")
 .save("/opt/workspace/data/delta_lake/nobel_prizes"))

In [0]:
# Query the partitioned table
df = spark.read.format("delta").load("/opt/workspace/data/delta_lake/nobel_prizes")
df.show()

In [0]:
spark.stop()